In [1]:
import os
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import re
import chromadb
from langchain_openai import OpenAIEmbeddings, OpenAI, ChatOpenAI
from langchain_chroma import Chroma
from dotenv import load_dotenv, find_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_community.document_loaders import UnstructuredHTMLLoader


# 1. Load and Split a Document

In [45]:
with open("documents/odyssey.txt", "r", encoding="utf-8") as f:
    text = f.read()

In [46]:
pattern = r"(?m)^\s*(BOOK\s+[IVXLCDM]+)\s*$"

In [2]:
_ = load_dotenv(find_dotenv())

In [3]:
OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

* TextLoader is used because we are working with .txt and not .pdf

In [54]:
loader = TextLoader("documents/odyssey.txt", encoding="utf-8")

In [55]:
documents = loader.load()

* I tried using 500 chunks at first, but it gave me around 1950 chunks. Considering this is a book, larger chunks could have more relevant information.

In [56]:
chunk_size = 1000

In [57]:
chunk_overlap = 200

In [58]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)

In [59]:
splited_documents = text_splitter.split_documents(documents) 

In [60]:
print("Total chunks:", len(splited_documents))

Total chunks: 974


* Show first 2 chunks with metadata:

In [11]:
print(splited_documents[:2])

[Document(metadata={'source': 'documents/odyssey.txt'}, page_content='\ufeffThe Odyssey\n\nby Homer\n\nPREFACE TO FIRST EDITION\n\n\nThis translation is intended to supplement a work entitled “The\nAuthoress of the Odyssey”, which I published in 1897. I could not give\nthe whole “Odyssey” in that book without making it unwieldy, I\ntherefore epitomised my translation, which was already completed and\nwhich I now publish in full.\n\nI shall not here argue the two main points dealt with in the work just\nmentioned; I have nothing either to add to, or to withdraw from, what I\nhave there written. The points in question are:\n\n(1) that the “Odyssey” was written entirely at, and drawn entirely\nfrom, the place now called Trapani on the West Coast of Sicily, alike\nas regards the Phaeacian and the Ithaca scenes; while the voyages of\nUlysses, when once he is within easy reach of Sicily, solve themselves\ninto a periplus of the island, practically from Trapani back to\nTrapani, via the Lipar

# 2. Generate Embeddings and Store in ChromaDB

In [61]:
PERSIST_DIRECTORY = "./chroma_langchain_db"
COLLECTION_NAME = "odyssey_collection"

In [62]:
if os.path.exists(PERSIST_DIRECTORY) and os.listdir(PERSIST_DIRECTORY): # If the folder exists and it is not empty
    print("Loading existing Chroma collection...")
    vector_store = Chroma(
        collection_name=COLLECTION_NAME,
        persist_directory=PERSIST_DIRECTORY,
        embedding_function=OpenAIEmbeddings()
    )
else:
    print("Creating new Chroma collection...")
    vector_store = Chroma.from_documents(
        documents=splited_documents,
        embedding=OpenAIEmbeddings(),
        collection_name=COLLECTION_NAME,
        persist_directory=PERSIST_DIRECTORY
    )

Loading existing Chroma collection...


In [63]:
retriever = vector_store.as_retriever()

In [64]:
print("Collection count:", vector_store._collection.count())

Collection count: 974


# 3. Implementation of a Basic RAG Chain

In [65]:
def format_docs(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)

In [66]:
llm = ChatOpenAI(model="gpt-4", temperature=0)

In [67]:
PROMPT = """
You are an assistant chatbot that is an expert on Homer's Odyssey. You will answer the user's questions ONLY based on the received context,  
DO NOT use external sources. In your answers, be precise and answer in a concise way (2-5 sentences).

- If context does not contain the answer, just say "I don't know".
- In your answer, include useful metadata if present in context.
            
Context: 
{context}

User question: 
{question}
"""

In [68]:
prompt = ChatPromptTemplate.from_template(PROMPT)

The first object recieves the input.
* In the first branch, it uses the input to call the retriever, bring the relevant documents and then format that. After that, it saves it on "context" variable.
* In the second branch, the RunnablePassthrough leaves the input intact, so the question is saved on "question" key.

After the object is complete, it inyects it into the prompt, it passes the prompt to the LLM, and then it formats the output.

In [69]:
rag_chain = ({"context": retriever | format_docs, "question": RunnablePassthrough()} 
            | prompt | llm | StrOutputParser()
)

In [70]:
questions = ["Who is Telemachus?",
            "Who is Penelope?",
            "Who is Athena helping in the story?",
            "How does Odysseus escape from the Cyclops?",
            "How does Odysseus reveal himself to the Cyclops?",
            "Why does Poseidon hate Odysseus?",
            "What happens when Odysseus reaches the island of the Sirens?",
            "How does Penelope test Odysseus when he returns home?",
            "How does Odysseus prove his identity at the end?",
            "What is Odysseus’ main struggle during his journey?",
            "What role do the gods play in Odysseus’ fate?",
            "How is loyalty portrayed in the Odyssey?",
            "Who was Shakespeare?"]

In [74]:
user_query = questions[10]

In [75]:
response = rag_chain.invoke(user_query)

In [76]:
print(response)

The gods play a significant role in Odysseus' fate. They have the power to detain him, as seen when the gods held him in Egypt because his hecatombs did not satisfy them. A goddess, Idothea, daughter to Proteus, saves him from starvation. Additionally, the goddess Athene expresses concern for Odysseus, who is suffering on a sea-girt isle, held captive by the daughter of the wizard Atlas. The gods also seem to have influence over the actions of others, as suggested by the belief that a god has told people that Ulysses is dead.


# 4. Mini Project: Q&A Chatbot on a Python Library

In [135]:
quickstart_file = UnstructuredHTMLLoader("documents/requests-quickstart.html")
quickstart_docs = quickstart_file.load()

In [136]:
for doc in quickstart_docs:
    doc.metadata["source"] = "quickstart" # Tag all the quickstart content

In [35]:
print(quickstart_docs)

[Document(metadata={'source': 'quickstart'}, page_content='Quickstart¶\n\nEager to get started? This page gives a good introduction in how to get started with Requests.\n\nFirst, make sure that:\n\nRequests is installed\n\nRequests is up-to-date\n\nLet’s get started with some simple examples.\n\nMake a Request¶\n\nMaking a request with Requests is very simple.\n\nBegin by importing the Requests module:\n\n>>> import requests\n\nNow, let’s try to get a webpage. For this example, let’s get GitHub’s public timeline:\n\n>>> r = requests.get(\'https://api.github.com/events\')\n\nNow, we have a Response object called r. We can get all the information we need from this object.\n\nRequests’ simple API means that all forms of HTTP request are as obvious. For example, this is how you make an HTTP POST request:\n\n>>> r = requests.post(\'https://httpbin.org/post\', data={\'key\': \'value\'})\n\nNice, right? What about the other HTTP request types: PUT, DELETE, HEAD and OPTIONS? These are all just

In [29]:
advanced_file = UnstructuredHTMLLoader("documents/requests-advanced.html")
advanced_docs = advanced_file.load()

In [137]:
for doc in advanced_docs:
    doc.metadata["source"] = "advanced" # Tag all the advanced content

In [37]:
print(advanced_docs)

[Document(metadata={'source': 'advanced'}, page_content='Advanced Usage¶\n\nThis document covers some of Requests more advanced features.\n\nSession Objects¶\n\nThe Session object allows you to persist certain parameters across requests. It also persists cookies across all requests made from the Session instance, and will use urllib3’s connection pooling. So if you’re making several requests to the same host, the underlying TCP connection will be reused, which can result in a significant performance increase (see HTTP persistent connection).\n\nA Session object has all the methods of the main Requests API.\n\nLet’s persist some cookies across requests:\n\ns = requests.Session()\n\ns.get(\'https://httpbin.org/cookies/set/sessioncookie/123456789\')\nr = s.get(\'https://httpbin.org/cookies\')\n\nprint(r.text)\n# \'{"cookies": {"sessioncookie": "123456789"}}\'\n\nSessions can also be used to provide default data to the request methods. This is done by providing data to the properties on a 

In [138]:
all_docs = quickstart_docs + advanced_docs # Unify all the quickstart and advanced content

In [39]:
print(all_docs)

[Document(metadata={'source': 'quickstart'}, page_content='Quickstart¶\n\nEager to get started? This page gives a good introduction in how to get started with Requests.\n\nFirst, make sure that:\n\nRequests is installed\n\nRequests is up-to-date\n\nLet’s get started with some simple examples.\n\nMake a Request¶\n\nMaking a request with Requests is very simple.\n\nBegin by importing the Requests module:\n\n>>> import requests\n\nNow, let’s try to get a webpage. For this example, let’s get GitHub’s public timeline:\n\n>>> r = requests.get(\'https://api.github.com/events\')\n\nNow, we have a Response object called r. We can get all the information we need from this object.\n\nRequests’ simple API means that all forms of HTTP request are as obvious. For example, this is how you make an HTTP POST request:\n\n>>> r = requests.post(\'https://httpbin.org/post\', data={\'key\': \'value\'})\n\nNice, right? What about the other HTTP request types: PUT, DELETE, HEAD and OPTIONS? These are all just

In [41]:
c_size = 1000
c_overlap = 200

In [42]:
text_splitter_requests = RecursiveCharacterTextSplitter(chunk_size=c_size, chunk_overlap=c_overlap)

In [43]:
splited_documents_requests = text_splitter_requests.split_documents(all_docs) 

In [83]:
print("Total chunks:", len(splited_documents_requests))

Total chunks: 70


In [51]:
PERSIST_DIRECTORY_REQUESTS = "./requests_langchain_db"
COLLECTION_NAME_REQUESTS = "requests_collection"

In [77]:
if os.path.exists(PERSIST_DIRECTORY_REQUESTS) and os.listdir(PERSIST_DIRECTORY_REQUESTS): # If the folder exists and it is not empty
    print("Loading existing Chroma collection...")
    vector_store_requests = Chroma(
        collection_name=COLLECTION_NAME_REQUESTS,
        persist_directory=PERSIST_DIRECTORY_REQUESTS,
        embedding_function=OpenAIEmbeddings()
    )
else:
    print("Creating new Chroma collection...")
    vector_store_requests = Chroma.from_documents(
        documents=splited_documents_requests,
        embedding=OpenAIEmbeddings(),
        collection_name=COLLECTION_NAME_REQUESTS,
        persist_directory=PERSIST_DIRECTORY_REQUESTS
    )

Loading existing Chroma collection...


In [78]:
retriever_requests = vector_store_requests.as_retriever()

In [79]:
print("Collection count:", vector_store_requests._collection.count())

Collection count: 70


In [121]:
PROMPT_REQUESTS = """
You are an assistant chatbot that is an expert on Requests Python Library. You will answer the user's questions ONLY based on the received context,  
DO NOT use external sources. In your answers, be precise and answer in a concise way (2-5 sentences).

- If context does not contain the answer, just say "I don't know".
- In your answer, mention whether the infromation comes from Quickstart or Advanced.

Response format:

- Answer: <YOUR_ANSWER>

- Source: <<ADVANCED OR QUICKSTART>> 
            
Context: 
{context}

User question: 
{question}
"""

In [122]:
prompt_requests = ChatPromptTemplate.from_template(PROMPT_REQUESTS)

In [123]:
rag_chain_requests = ({"context": retriever_requests | format_docs, "question": RunnablePassthrough()} 
            | prompt_requests | llm | StrOutputParser()
)

In [131]:
questions_requests = ["Tell me about timeout parameter",
            "What happens if multiple IP addresses exist for a domain name regarding timeouts?",
            "Does requests support HTTP/3?", # Not in the documentation
            "How do I use asyncio with requests?", # Not in the documentation
            "How do I set retries with exponential backoff?", # Not in the documentation
            "How can I read a JSON response?"
            ]

In [132]:
user_query_requests = questions_requests[5]

In [133]:
response_requests = rag_chain_requests.invoke(user_query_requests)

In [134]:
print(response_requests)

Answer: You can read a JSON response using the `r.json()` method. This method parses the JSON response into Python objects. If the JSON decoding fails, `r.json()` raises an exception. For example, if the response gets a 204 (No Content), or if the response contains invalid JSON, attempting `r.json()` raises `requests.exceptions.JSONDecodeError`.

Source: Quickstart
